# ArticleTagging — Setup & Benchmark (Qwen3-VL-8B)
Run this notebook on Lightning AI to set up the environment and benchmark Qwen3-VL-8B.

## 1. Clone Repo & Run Setup

In [ ]:
!git clone https://github.com/Vutiot/ArticleTagging.git
%cd ArticleTagging

In [ ]:
!bash scripts/lightning_setup.sh

## 2. Verify GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 3. Get Dataset
Upload `fashion-data.tar.gz` via the Kaggle sidebar, or download from Kaggle API.

In [ ]:
# Option A: If you uploaded fashion-data.tar.gz
# !tar xzf /kaggle/input/fashion-data/fashion-data.tar.gz

# Option B: Download from Kaggle datasets API
# !kaggle datasets download paramaggarwal/fashion-product-images-small -p data/raw/fashion/ --unzip

In [ ]:
# Verify data
!ls data/processed/fashion/ 2>/dev/null || echo "No processed data yet — run prepare step"
!ls data/raw/fashion/images/ 2>/dev/null | head -5 || echo "No images yet"

## 4. Train

In [ ]:
# Quick smoke test (10 steps)
!python scripts/benchmark_qwen3vl_8b.py --phase train --max-steps 10

In [ ]:
# Full training (3 epochs)
!python scripts/benchmark_qwen3vl_8b.py --phase train

## 4.1 Monitor Training
Poll the training log to track progress while training runs in the background.

In [ ]:
# Re-run this cell to check training progress
from pathlib import Path

log_path = Path("models/fashion-qwen3vl-8b/training.log")
if log_path.exists():
    lines = log_path.read_text().strip().splitlines()
    for line in lines[-20:]:
        print(line)
    print(f"\n--- {len(lines)} total log lines ---")
else:
    print("Training log not found yet. Start training first.")

# Check for checkpoints
ckpt_dir = Path("models/fashion-qwen3vl-8b")
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob("checkpoint-*"))
    if ckpts:
        print(f"\nCheckpoints: {len(ckpts)}")
        for c in ckpts[-3:]:
            print(f"  {c.name}")

# VRAM usage
import torch
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"\nVRAM: {alloc:.1f} GB allocated, {peak:.1f} GB peak")

## 5. Evaluate

In [ ]:
!python scripts/benchmark_qwen3vl_8b.py --phase eval

In [ ]:
# View results
import json
from pathlib import Path

result_path = Path("reports/v3_qwen3vl_8b/eval_result.json")
if result_path.exists():
    result = json.loads(result_path.read_text())
    print(f"Exact Match: {result['exact_match']:.1%}")
    print("Per-attribute:")
    for attr, acc in result['per_attribute'].items():
        print(f"  {attr}: {acc:.1%}")
else:
    print("No results yet — run eval first")

## 6. Latency Benchmark (vLLM)

In [ ]:
!python scripts/benchmark_qwen3vl_8b.py --phase latency